In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller,kpss
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
plt.style.use("default")

In [ ]:
df = pd.read_excel("/content/Master_Dataset.xlsx")

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values("Date")

df.set_index("Date", inplace=True)

df.head()

#||| **USDINR** |||

In [ ]:
series = pd.to_numeric(

    df["USDINR"],

    errors="coerce"

)

series = series.dropna()

series.name = "USDINR"

series.head()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(

    series,

    lw=2

)

plt.title("USDINR Exchange Rate")

plt.xlabel("Year")

plt.ylabel("USDINR")

plt.grid(alpha=.3)

plt.show()

In [ ]:
rolling_mean = series.rolling(12).mean()

rolling_std = series.rolling(12).std()

plt.figure(figsize=(15,5))

plt.plot(series,label="USDINR")

plt.plot(

    rolling_mean,

    label="12M Rolling Mean"

)

plt.plot(

    rolling_std,

    label="12M Rolling Std"

)

plt.legend()

plt.grid(alpha=.3)

plt.title("Rolling Statistics")

plt.show()

In [ ]:
plt.figure(figsize=(6,5))

plt.hist(

    series,

    bins=30,

    edgecolor="black"

)

plt.title("Distribution of USDINR")

plt.grid(alpha=.3)

plt.show()

In [ ]:
result = adfuller(series)

print("ADF Statistic :",result[0])

print("P-value :",result[1])

print()

print("Critical Values")

for k,v in result[4].items():

    print(k,":",v)

In [ ]:
result = kpss(

    series,

    regression="c"

)

print("KPSS Statistic :",result[0])

print("P-value :",result[1])

In [ ]:
series_diff = series.diff().dropna()

plt.figure(figsize=(15,4))

plt.plot(series_diff)

plt.axhline(

0,

color="red",

ls="--"

)

plt.title("First Difference")

plt.grid(alpha=.3)

plt.show()

In [ ]:
print(

adfuller(series_diff)[1]

)

In [ ]:
plot_acf(

    series_diff,

    lags=30

)

plot_pacf(

    series_diff,

    lags=30

)

plt.show()



In [ ]:
train = series.iloc[:-24]

test = series.iloc[-24:]

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

results = []

best_aic = np.inf
best_model = None
best_order = None

for p in range(4):

    for d in [1]:

        for q in range(4):

            try:

                model = ARIMA(

                    train,

                    order=(p,d,q)

                )

                fit = model.fit()

                forecast = fit.forecast(len(test))

                rmse = np.sqrt(mean_squared_error(test, forecast))

                mae = mean_absolute_error(test, forecast)

                mape = mean_absolute_percentage_error(test, forecast)*100

                results.append({

                    "Order":(p,d,q),

                    "AIC":fit.aic,

                    "BIC":fit.bic,

                    "RMSE":rmse,

                    "MAE":mae,

                    "MAPE":mape

                })

                if fit.aic < best_aic:

                    best_aic = fit.aic

                    best_model = fit

                    best_order = (p,d,q)

            except:

                continue

arima_table = pd.DataFrame(results)

arima_table.sort_values("RMSE")

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
model = ARIMA(
    train,
    order = (1,1,1)
)

fit = model.fit()
print(fit.summary())

In [ ]:
train_fit = fit.fittedvalues
forecast = fit.forecast(

    steps=len(test)

)

In [ ]:
from sklearn.metrics import (

    mean_squared_error,

    mean_absolute_error,

    mean_absolute_percentage_error

)

rmse = np.sqrt(

    mean_squared_error(

        test,

        forecast

    )

)

mae = mean_absolute_error(

    test,

    forecast

)

mape = mean_absolute_percentage_error(

    test,

    forecast

)*100

print("RMSE :",rmse)

print("MAE :",mae)

print("MAPE :",mape)

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(

    train,

    label="Train",

    lw=2

)

plt.plot(

    train.index[1:],

    train_fit.iloc[1:],

    label="Train Fitted",

    lw=2

)

plt.plot(

    test,

    label="Actual",

    lw=2

)

plt.plot(

    test.index,

    forecast,

    "--",

    lw=2,

    label="Forecast"

)

plt.title("ARIMA Forecast : USDINR")

plt.xlabel("Year")

plt.ylabel("USDINR")

plt.grid(alpha=.3)

plt.legend()

plt.show()

In [ ]:
residuals = fit.resid

plt.figure(figsize=(15,4))

plt.plot(

    residuals

)

plt.axhline(

0,

color="red",

ls="--"

)

plt.title("Residuals")

plt.grid(alpha=.3)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(

    residuals,

    bins=30,

    edgecolor="black"

)

plt.title("Residual Distribution")

plt.grid(alpha=.3)

plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(

    residuals,

    lags=30

)

plt.show()

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

lb = acorr_ljungbox(

    residuals,

    lags=[10],

    return_df=True

)

lb

In [ ]:
future = fit.get_forecast(steps=48)

forecast = future.predicted_mean

confidence = future.conf_int(alpha=0.05)

In [ ]:
print(series.index.min())
print(series.index.max())

print(train.index.min())
print(train.index.max())

In [ ]:
print(len(series))
print(len(train))
print(len(test))

In [ ]:
confidence.head()

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(series, label="Historical")

plt.plot(
    forecast.index,
    forecast,
    "r--",
    label="Forecast"
)

plt.fill_between(
    forecast.index,
    confidence.iloc[:,0],
    confidence.iloc[:,1],
    color="red",
    alpha=0.25,
    label="95% Confidence Interval"
)

plt.legend()

plt.grid(alpha=0.3)

plt.show()

In [ ]:
forecast_df_USDINR = forecast.copy()

In [ ]:
print(forecast_df_USDINR.head())
print(forecast_df_USDINR.tail())

In [ ]:
df_confidence_USDINR = confidence.copy()

# ||| **Inflation** |||

In [ ]:
series = pd.to_numeric(
    df["Inflation_YoY"],
    errors="coerce"
)

series = series.dropna()

series.name = "Inflation_YoY"

series.head()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(
    series,
    lw=2
)

plt.title("Inflation YoY")

plt.xlabel("Year")

plt.ylabel("Inflation (%)")

plt.grid(alpha=.3)

plt.show()

In [ ]:
rolling_mean = series.rolling(12).mean()

rolling_std = series.rolling(12).std()

plt.figure(figsize=(15,5))

plt.plot(series,label="Inflation")

plt.plot(
    rolling_mean,
    label="12M Rolling Mean"
)

plt.plot(
    rolling_std,
    label="12M Rolling Std"
)

plt.legend()

plt.grid(alpha=.3)

plt.title("Rolling Statistics")

plt.show()

In [ ]:
plt.figure(figsize=(6,5))

plt.hist(
    series,
    bins=25,
    edgecolor="black"
)

plt.title("Distribution of Inflation")

plt.grid(alpha=.3)

plt.show()

In [ ]:
result = adfuller(series)

print("ADF Statistic :",result[0])

print("P-value :",result[1])

print()

for k,v in result[4].items():

    print(k,":",v)

In [ ]:
result = kpss(
    series,
    regression="c"
)

print("KPSS Statistic :",result[0])

print("P-value :",result[1])

In [ ]:
series_diff = series.diff().dropna()

plt.figure(figsize=(15,4))

plt.plot(series_diff)

plt.axhline(
    0,
    color="red",
    ls="--"
)

plt.title("First Difference")

plt.grid(alpha=.3)

plt.show()

In [ ]:
print(adfuller(series_diff)[1])

In [ ]:
plot_acf(
    series_diff if adfuller(series)[1] > 0.05 else series,
    lags=30
)

plt.show()

In [ ]:
plot_pacf(
    series_diff if adfuller(series)[1] > 0.05 else series,
    lags=30
)

plt.show()

In [ ]:
train = series.iloc[:-24]

test = series.iloc[-24:]

d = 0 if adfuller(series)[1] < 0.05 else 1

from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

results = []

best_aic = np.inf
best_model = None
best_order = None

for p in range(4):

    for q in range(4):

        try:

            model = ARIMA(
                train,
                order=(p,d,q)
            )

            fit = model.fit()

            forecast = fit.forecast(len(test))

            rmse = np.sqrt(mean_squared_error(test,forecast))

            mae = mean_absolute_error(test,forecast)

            mape = mean_absolute_percentage_error(test,forecast)*100

            results.append({

                "Order":(p,d,q),

                "AIC":fit.aic,

                "BIC":fit.bic,

                "RMSE":rmse,

                "MAE":mae,

                "MAPE":mape

            })

            if fit.aic < best_aic:

                best_aic = fit.aic

                best_model = fit

                best_order = (p,d,q)

        except:

            continue

arima_table = pd.DataFrame(results)

arima_table.sort_values("RMSE")

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

import warnings
warnings.filterwarnings("ignore")

# d from ADF
d = 0 if adfuller(series)[1] < 0.05 else 1

# Seasonal period
s = 12

# Non-seasonal orders to test
p_values = range(0,4)
q_values = range(0,4)

# Seasonal orders
seasonal_orders = [

    (0,0,0,s),

    (1,0,0,s),

    (0,0,1,s),

    (1,0,1,s),

    (0,1,0,s),

    (1,1,0,s),

    (0,1,1,s),

    (1,1,1,s)

]

results = []

best_aic = np.inf
best_model = None
best_order = None
best_seasonal = None

for p in p_values:

    for q in q_values:

        for seasonal in seasonal_orders:

            try:

                model = SARIMAX(

                    train,

                    order=(p,d,q),

                    seasonal_order=seasonal,

                    enforce_stationarity=False,

                    enforce_invertibility=False

                )

                fit = model.fit(disp=False)

                forecast = fit.forecast(len(test))

                rmse = np.sqrt(mean_squared_error(test,forecast))

                mae = mean_absolute_error(test,forecast)

                mape = mean_absolute_percentage_error(test,forecast)*100

                results.append({

                    "Order":(p,d,q),

                    "Seasonal Order":seasonal,

                    "AIC":fit.aic,

                    "BIC":fit.bic,

                    "RMSE":rmse,

                    "MAE":mae,

                    "MAPE":mape

                })

                if fit.aic < best_aic:

                    best_aic = fit.aic

                    best_model = fit

                    best_order = (p,d,q)

                    best_seasonal = seasonal

            except:

                continue

sarima_table = pd.DataFrame(results)

sarima_table.sort_values("RMSE")

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

fit = SARIMAX(

    train,

    order=(0,1,1),

    seasonal_order=(1,1,0,12),

    enforce_stationarity=False,

    enforce_invertibility=False

).fit(disp=False)

print(fit.summary())

In [ ]:
train_fit = fit.fittedvalues

forecast = fit.forecast(

    steps=len(test)

)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

rmse = np.sqrt(
    mean_squared_error(test,forecast)
)

mae = mean_absolute_error(
    test,
    forecast
)

mape = mean_absolute_percentage_error(
    test,
    forecast
)*100

print("RMSE :",rmse)
print("MAE :",mae)
print("MAPE :",mape)

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(
    train,
    label="Train",
    lw=2
)

plt.plot(
    train.index[13:],
    train_fit[13:],
    label="Train Fitted",
    lw=2
)

plt.plot(
    test,
    label="Actual",
    lw=2
)

plt.plot(
    test.index,
    forecast,
    "r--",
    lw=2,
    label="Forecast"
)

plt.title("SARIMA Forecast : Inflation")

plt.xlabel("Year")

plt.ylabel("Inflation (%)")

plt.grid(alpha=0.3)

plt.legend()

plt.show()

In [ ]:
residuals = fit.resid

plt.figure(figsize=(15,4))

plt.plot(residuals)

plt.axhline(
    0,
    color="red",
    linestyle="--"
)

plt.title("SARIMA Residuals")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
plt.figure(figsize=(7,5))

plt.hist(
    residuals,
    bins=25,
    edgecolor="black"
)

plt.title("Residual Distribution")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(
    residuals,
    lags=30
)

plt.show()

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

lb = acorr_ljungbox(

    residuals,

    lags=[10],

    return_df=True

)

lb

In [ ]:
future = fit.get_forecast(

    steps=48

)

forecast = future.predicted_mean

confidence = future.conf_int()

In [ ]:
future_dates = pd.date_range(

    start=series.index[-1] + pd.offsets.MonthBegin(1),

    periods=48,

    freq="MS"

)

plt.figure(figsize=(16,6))

plt.plot(
    series,
    label="Historical",
    lw=2
)

plt.plot(
    future_dates,
    forecast,
    "r--",
    lw=2,
    label="Forecast"
)

plt.fill_between(
    future_dates,
    confidence.iloc[:,0],
    confidence.iloc[:,1],
    color="red",
    alpha=0.25,
    label="95% Confidence Interval"
)

plt.title("Inflation Forecast (Next 48 Months)")

plt.xlabel("Year")

plt.ylabel("Inflation (%)")

plt.grid(alpha=0.3)

plt.legend()

plt.show()

In [ ]:
forecast_df_Inflation = pd.DataFrame({

    "Date":future_dates,

    "Inflation_Predicted":forecast.values,

    "Inflation_Lower_CI":confidence.iloc[:,0].values,

    "Inflation_Upper_CI":confidence.iloc[:,1].values

})

forecast_df_Inflation.to_csv(

    "Inflation_Forecast.csv",

    index=False

)

forecast_df_Inflation.head()

In [ ]:
forecast_df_Inflation

# ||| **FPI** |||

In [ ]:
series = pd.to_numeric(
    df["FPI_Net_INR_Crore"],
    errors="coerce"
).dropna()

series = series.asfreq("MS")

series.head()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(series,label="FPI")

plt.plot(
    series.rolling(12).mean(),
    label="12M Rolling Mean"
)

plt.plot(
    series.rolling(12).std(),
    label="12M Rolling Std"
)

plt.legend()

plt.grid(alpha=0.3)

plt.title("Rolling Statistics")

plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(series.dropna())

print("ADF Statistic :",result[0])

print("P-value :",result[1])

for k,v in result[4].items():
    print(k,":",v)

In [ ]:
kpss_result = kpss(series.dropna())

print("KPSS Statistic :",kpss_result[0])

print("P-value :",kpss_result[1])

In [ ]:
series_diff = series.diff().dropna()

plt.figure(figsize=(15,4))

plt.plot(series_diff)

plt.axhline(0,color="red",ls="--")

plt.title("First Difference")

plt.grid(alpha=0.3)

plt.show()

print(adfuller(series_diff)[1])

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf

plot_acf(
    series_diff if adfuller(series)[1]>0.05 else series,
    lags=30
)

plt.show()

plot_pacf(
    series_diff if adfuller(series)[1]>0.05 else series,
    lags=30
)

plt.show()

In [ ]:
train = series.iloc[:-24]

test = series.iloc[-24:]



In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

d = 0 if adfuller(series)[1] < 0.05 else 1

results = []

best_aic = np.inf
best_model = None
best_order = None

for p in range(3):

    for q in range(3):

        try:

            fit = ARIMA(
                train,
                order=(p,d,q)
            ).fit()

            forecast = fit.forecast(len(test))

            rmse = np.sqrt(mean_squared_error(test,forecast))
            mae = mean_absolute_error(test,forecast)
            mape = mean_absolute_percentage_error(test,forecast)*100

            results.append({

                "Order":(p,d,q),
                "AIC":fit.aic,
                "BIC":fit.bic,
                "RMSE":rmse,
                "MAE":mae,
                "MAPE":mape

            })

            if rmse < best_aic:

                best_aic = rmse
                best_model = fit
                best_order = (p,d,q)

        except:
            pass

arima_table = pd.DataFrame(results)

arima_table.sort_values("RMSE")

In [ ]:
plt.figure(figsize=(15,5))
plt.plot(series)
plt.grid(alpha=.3)
plt.show()

In [ ]:
series.describe()

In [ ]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method="yeo-johnson")

series_yj = pd.Series(
    pt.fit_transform(series.values.reshape(-1,1)).ravel(),
    index=series.index
)

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(series_yj)

plt.title("Yeo-Johnson Transformed FPI")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

print(adfuller(series_yj)[1])

In [ ]:
train = series_yj.iloc[:-24]

test = series_yj.iloc[-24:]

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

d = 0 if adfuller(series)[1] < 0.05 else 1

results = []

best_aic = np.inf
best_model = None
best_order = None

for p in range(3):

    for q in range(3):

        try:

            fit = ARIMA(
                train,
                order=(p,d,q)
            ).fit()

            forecast = fit.forecast(len(test))

            rmse = np.sqrt(mean_squared_error(test,forecast))
            mae = mean_absolute_error(test,forecast)
            mape = mean_absolute_percentage_error(test,forecast)*100

            results.append({

                "Order":(p,d,q),
                "AIC":fit.aic,
                "BIC":fit.bic,
                "RMSE":rmse,
                "MAE":mae,
                "MAPE":mape

            })

            if rmse < best_aic:

                best_aic = rmse
                best_model = fit
                best_order = (p,d,q)

        except:
            pass

arima_table = pd.DataFrame(results)

arima_table.sort_values("RMSE")

In [ ]:
model = ARIMA(
    train,
    order = (1,0,2)
)
fit = model.fit()
print(fit.summary())

In [ ]:
train_fit = fit.fittedvalues

forecast = fit.forecast(
    steps=len(test)
)

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

rmse = np.sqrt(mean_squared_error(test, forecast))

mae = mean_absolute_error(test, forecast)

mape = mean_absolute_percentage_error(
    test,
    forecast
) * 100

print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"MAPE : {mape:.2f}%")

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(
    train,
    label="Train",
    lw=2
)

plt.plot(
    train.index,
    train_fit,
    label="Train Fitted",
    lw=2
)

plt.plot(
    test,
    label="Actual",
    lw=2
)

plt.plot(
    test.index,
    forecast,
    "--",
    lw=2,
    label="Forecast"
)

plt.title("ARIMA Forecast : FPI (Yeo-Johnson)")

plt.xlabel("Year")

plt.ylabel("Transformed FPI")

plt.grid(alpha=.3)

plt.legend()

plt.show()

In [ ]:
residuals = fit.resid

plt.figure(figsize=(15,5))

plt.plot(residuals)

plt.axhline(
    0,
    color="red",
    ls="--"
)

plt.title("Residuals")

plt.grid(alpha=.3)

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    residuals,
    bins=25,
    edgecolor="black"
)

plt.title("Residual Distribution")

plt.grid(alpha=.3)

plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(
    residuals,
    lags=30
)

plt.show()

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

lb = acorr_ljungbox(
    residuals,
    lags=[10],
    return_df=True
)

print(lb)

In [ ]:
fpi_results = {

    "Model":"ARIMA (Yeo-Johnson)",

    "Order":(1,0,2),

    "AIC":fit.aic,

    "BIC":fit.bic,

    "RMSE":rmse,

    "MAE":mae,

    "MAPE":mape

}

fpi_results

In [ ]:
final_model = ARIMA(

    series_yj,

    order=(1,0,2)

)

final_fit = final_model.fit()

In [ ]:
future = final_fit.get_forecast(
    steps=48
)

forecast_yj = future.predicted_mean

confidence = future.conf_int(alpha=0.05)

In [ ]:
forecast_original = pt.inverse_transform(
    forecast_yj.values.reshape(-1,1)
).ravel()

lower_original = pt.inverse_transform(
    confidence.iloc[:,0].values.reshape(-1,1)
).ravel()

upper_original = pt.inverse_transform(
    confidence.iloc[:,1].values.reshape(-1,1)
).ravel()

In [ ]:
forecast_df_FPI = pd.DataFrame({

    "Date":forecast_yj.index,

    "FPI_Predicted":forecast_original,

    "FPI_Lower_CI":lower_original,

    "FPI_Upper_CI":upper_original

})

forecast_df_FPI.head()

In [ ]:
from statsmodels.stats.diagnostic import het_arch

arch_test = het_arch(residuals)

print("LM Statistic :", arch_test[0])
print("P-value      :", arch_test[1])

In [ ]:
!pip install arch

In [ ]:
from arch import arch_model
results = []

best_aic = np.inf
best_model = None

for p in [1,2]:

    for q in [1,2]:

        try:

            model = arch_model(

                residuals,

                mean="Zero",

                vol="GARCH",

                p=p,

                q=q,

                dist="normal"

            )

            fit = model.fit(disp="off")

            results.append({

                "Model":f"GARCH({p},{q})",

                "AIC":fit.aic,

                "BIC":fit.bic,

                "LogLik":fit.loglikelihood

            })

            if fit.aic < best_aic:

                best_aic = fit.aic

                best_model = fit

                best_order = (p,q)

        except:
            pass

garch_table = pd.DataFrame(results)

garch_table.sort_values("AIC")

In [ ]:
print(best_order)

print(best_model.summary())

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(
    best_model.conditional_volatility,
    label="Conditional Volatility"
)

plt.title("Estimated Volatility")

plt.grid(alpha=0.3)

plt.legend()

plt.show()

In [ ]:
std_resid = best_model.std_resid

plt.figure(figsize=(15,5))

plt.plot(std_resid)

plt.axhline(0,color="red",ls="--")

plt.title("Standardized Residuals")

plt.grid(alpha=0.3)

plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(
    std_resid,
    lags=30
)

plt.show()

In [ ]:
from statsmodels.stats.diagnostic import het_arch

lm = het_arch(std_resid)

print("LM Statistic :", lm[0])

print("P-value :", lm[1])

In [ ]:
forecast = best_model.forecast(horizon=48)

variance = forecast.variance.iloc[-1]

volatility = np.sqrt(variance)

volatility

In [ ]:
future_mean = final_fit.get_forecast(steps=48)

mean_forecast = future_mean.predicted_mean

garch_forecast = best_model.forecast(horizon=48)

variance = garch_forecast.variance.iloc[-1]

volatility = np.sqrt(variance)

lower = mean_forecast - 1.96*volatility.values

upper = mean_forecast + 1.96*volatility.values

forecast_df = pd.DataFrame({

    "Forecast":mean_forecast,

    "Volatility":volatility.values,

    "Lower_CI":lower,

    "Upper_CI":upper

})

forecast_df.head()

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(
    series_yj,
    label="Historical"
)

plt.plot(
    mean_forecast.index,
    mean_forecast,
    "r--",
    lw=2,
    label="ARIMA Mean Forecast"
)

plt.fill_between(

    mean_forecast.index,

    lower,

    upper,

    color="red",

    alpha=.25,

    label="95% Uncertainty Band"

)

plt.title("ARIMA + GARCH Forecast")

plt.grid(alpha=.3)

plt.legend()

plt.show()

In [ ]:
forecast_original = pt.inverse_transform(
    mean_forecast.values.reshape(-1,1)
).ravel()

lower_original = pt.inverse_transform(
    lower.values.reshape(-1,1)
).ravel()

upper_original = pt.inverse_transform(
    upper.values.reshape(-1,1)
).ravel()

In [ ]:
forecast_df_FPI = pd.DataFrame({

    "Date":mean_forecast.index,

    "Forecast":forecast_original,

    "Lower_CI":lower_original,

    "Upper_CI":upper_original

})

In [ ]:
forecast_df_FPI.head()

# ||| **Gold Reserves** |||

In [ ]:
series = pd.to_numeric(
    df["Gold_Reservs_USD_mn"],
    errors="coerce"
).dropna()

series = series.asfreq("MS")

series.head()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(
    series,
    label="Gold Reserves"
)

plt.plot(
    series.rolling(12).mean(),
    label="12M Rolling Mean"
)

plt.plot(
    series.rolling(12).std(),
    label="12M Rolling Std"
)

plt.title("Gold Reserves (USD Million)")

plt.grid(alpha=0.3)

plt.legend()

plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(series)

print("ADF Statistic :", result[0])
print("P-value :", result[1])

for k,v in result[4].items():
    print(k,":",v)

In [ ]:
series_diff = series.diff().dropna()

plt.figure(figsize=(15,4))

plt.plot(series_diff)

plt.axhline(
    0,
    color="red",
    ls="--"
)

plt.title("First Difference")

plt.grid(alpha=0.3)

plt.show()

print(
    "ADF after Difference:",
    adfuller(series_diff)[1]
)

In [ ]:
series_diff2 = series.diff().diff().dropna()

plt.figure(figsize=(15,4))

plt.plot(series_diff2)

plt.axhline(
    0,
    color="red",
    ls="--"
)

plt.title("Second Difference")

plt.grid(alpha=0.3)

plt.show()

print(
    "ADF after Second Difference:",
    adfuller(series_diff2)[1]
)

In [ ]:
from statsmodels.graphics.tsaplots import (
    plot_acf,
    plot_pacf
)

if adfuller(series)[1] < 0.05:

    model_series = series

elif adfuller(series_diff)[1] < 0.05:

    model_series = series_diff

else:

    model_series = series_diff2

plot_acf(
    model_series,
    lags=30
)

plt.show()

plot_pacf(
    model_series,
    lags=30
)

plt.show()

In [ ]:
train = series.iloc[:-24]

test = series.iloc[-24:]

print(len(train))

print(len(test))

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

d = 2

results = []

best_rmse = np.inf
best_model = None
best_order = None

for p in range(4):

    for q in range(4):

        try:

            fit = ARIMA(
                train,
                order=(p,d,q)
            ).fit()

            forecast = fit.forecast(
                steps=len(test)
            )

            rmse = np.sqrt(mean_squared_error(test,forecast))

            mae = mean_absolute_error(test,forecast)

            mape = mean_absolute_percentage_error(
                test,
                forecast
            )*100

            results.append({

                "Order":(p,d,q),

                "AIC":fit.aic,

                "BIC":fit.bic,

                "RMSE":rmse,

                "MAE":mae,

                "MAPE":mape

            })

            if rmse < best_rmse:

                best_rmse = rmse

                best_model = fit

                best_order = (p,d,q)

        except:
            pass

arima_table = pd.DataFrame(results)

arima_table.sort_values("RMSE")

In [ ]:
model = ARIMA(

    train,

    order=(2,2,0)

)

fit = model.fit()

print(fit.summary())

In [ ]:
train_fit = fit.fittedvalues

forecast = fit.forecast(
    steps=len(test)
)

In [ ]:
rmse = np.sqrt(mean_squared_error(test,forecast))

mae = mean_absolute_error(test,forecast)

mape = mean_absolute_percentage_error(
    test,
    forecast
)*100

print("RMSE :",rmse)

print("MAE :",mae)

print("MAPE :",mape)

In [ ]:
train_fit = fit.fittedvalues

forecast = fit.forecast(
    steps=len(test)
)

plt.figure(figsize=(16,6))

plt.plot(train,
         label="Train",
         lw=2)

plt.plot(train.index[2:],
         train_fit.iloc[2:],
         label="Train Fitted",
         lw=2)

plt.plot(test,
         label="Actual",
         lw=2)

plt.plot(test.index,
         forecast,
         "--",
         lw=2,
         label="Forecast")

plt.title("ARIMA Forecast : Gold Reserves")
plt.xlabel("Year")
plt.ylabel("Gold Reserves (USD mn)")

plt.grid(alpha=.3)
plt.legend()

plt.show()

In [ ]:
residuals = fit.resid

plt.figure(figsize=(15,5))

plt.plot(residuals)

plt.axhline(
    0,
    color="red",
    linestyle="--"
)

plt.title("Residuals")
plt.grid(alpha=.3)

plt.show()

In [ ]:
plt.figure(figsize=(7,5))

plt.hist(
    residuals,
    bins=30,
    edgecolor="black"
)

plt.title("Residual Distribution")

plt.grid(alpha=.3)

plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(
    residuals,
    lags=30
)

plt.show()

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

lb = acorr_ljungbox(
    residuals,
    lags=[10],
    return_df=True
)

lb

In [ ]:
from statsmodels.stats.diagnostic import het_arch

arch = het_arch(residuals)

print("LM Statistic :", arch[0])
print("P-value      :", arch[1])

In [ ]:
from arch import arch_model

garch = arch_model(
    residuals,
    mean="Zero",
    vol="GARCH",
    p=1,
    q=1,
    dist="normal"
)

garch_fit = garch.fit(disp="off")

print(garch_fit.summary())

In [ ]:
garch_forecast = garch_fit.forecast(
    horizon=48,
    reindex=False
)

future_sigma = np.sqrt(
    garch_forecast.variance.values[-1]
)

future_sigma

In [ ]:
future = fit.get_forecast(steps=48)

mean_forecast = future.predicted_mean

In [ ]:
lower = mean_forecast - 1.96 * future_sigma
upper = mean_forecast + 1.96 * future_sigma

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(
    series,
    label="Historical",
    lw=2
)

plt.plot(
    mean_forecast.index,
    mean_forecast,
    "r--",
    lw=2,
    label="Forecast"
)

plt.fill_between(
    mean_forecast.index,
    lower,
    upper,
    color="red",
    alpha=.25,
    label="95% Uncertainty"
)

plt.title("ARIMA + GARCH Forecast : Gold Reserves")

plt.grid(alpha=.3)

plt.legend()

plt.show()

In [ ]:
forecast_df_GoldReserves = mean_forecast.copy()

confidence_df_GoldReserves = pd.DataFrame({
    "Lower": lower,
    "Upper": upper
})

forecast_df_GoldReserves

# ||| **GDP, current prices** |||

In [ ]:
series = pd.to_numeric(
    df["GDP, current prices (Billions of U.S. dollars)"],
    errors="coerce"
).dropna()

series = series.asfreq("MS")

series.head()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(series,label="GDP")
plt.plot(series.rolling(12).mean(),label="12M Mean")
plt.plot(series.rolling(12).std(),label="12M Std")

plt.title("GDP")
plt.grid(alpha=.3)
plt.legend()

plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(series)

print("ADF:",result[0])
print("P-value:",result[1])

for k,v in result[4].items():
    print(k,v)

In [ ]:
series_diff = series.diff().dropna()

plt.figure(figsize=(15,4))
plt.plot(series_diff)
plt.axhline(0,color="red",ls="--")
plt.title("First Difference")
plt.grid(alpha=.3)
plt.show()

print(adfuller(series_diff)[1])

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf

if adfuller(series)[1] < .05:
    model_series = series

elif adfuller(series_diff)[1] < .05:
    model_series = series_diff

else:
    model_series = series_diff2

plot_acf(model_series,lags=30)
plt.show()

plot_pacf(model_series,lags=30)
plt.show()

In [ ]:
train = series.iloc[:-24]

test = series.iloc[-24:]

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

results = []

best_rmse = np.inf
best_model = None
best_order = None
best_seasonal = None

for p in range(3):
    for q in range(3):
        for P in range(2):
            for Q in range(2):

                try:

                    model = SARIMAX(
                        train,
                        order=(p,1,q),
                        seasonal_order=(P,1,Q,12),
                        enforce_stationarity=False,
                        enforce_invertibility=False
                    )

                    fit = model.fit(disp=False)

                    forecast = fit.forecast(
                        steps=len(test)
                    )

                    rmse = np.sqrt(
                        mean_squared_error(
                            test,
                            forecast
                        )
                    )

                    mae = mean_absolute_error(
                        test,
                        forecast
                    )

                    mape = mean_absolute_percentage_error(
                        test,
                        forecast
                    )*100

                    results.append({

                        "Order":(p,1,q),

                        "Seasonal":(P,1,Q,12),

                        "AIC":fit.aic,

                        "BIC":fit.bic,

                        "RMSE":rmse,

                        "MAE":mae,

                        "MAPE":mape

                    })

                    if rmse < best_rmse:

                        best_rmse = rmse
                        best_model = fit
                        best_order = (p,1,q)
                        best_seasonal = (P,1,Q,12)

                except:
                    pass

sarima_table = pd.DataFrame(results)

sarima_table.sort_values("RMSE")

In [ ]:
model = SARIMAX(

    train,

    order=(0,1,0),

    seasonal_order=(1,1,0,12),

    enforce_stationarity=False,

    enforce_invertibility=False

)

fit = model.fit()

print(fit.summary())

In [ ]:
train_fit = fit.fittedvalues

forecast = fit.forecast(
    steps=len(test)
)

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(
    train,
    label="Train",
    lw=2
)

plt.plot(
    train.index[13:],
    train_fit.iloc[13:],
    label="Train Fitted",
    lw=2
)

plt.plot(
    test,
    label="Actual",
    lw=2
)

plt.plot(
    test.index,
    forecast,
    "r--",
    label="Forecast",
    lw=2
)

plt.title("SARIMA Forecast : GDP")

plt.grid(alpha=.3)

plt.legend()

plt.show()

In [ ]:
residuals = fit.resid

plt.figure(figsize=(15,5))

plt.plot(residuals)

plt.axhline(
    0,
    color="red",
    ls="--"
)

plt.title("Residuals")

plt.grid(alpha=.3)

plt.show()

In [ ]:
plt.figure(figsize=(6,4))

plt.hist(
    residuals,
    bins=25
)

plt.title("Residual Distribution")

plt.grid(alpha=.3)

plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(
    residuals,
    lags=30
)

plt.show()

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

lb = acorr_ljungbox(
    residuals,
    lags=[10],
    return_df=True
)

lb

In [ ]:
from statsmodels.stats.diagnostic import het_arch

arch = het_arch(residuals)

print("LM Statistic :",arch[0])
print("P-value :",arch[1])

In [ ]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

rmse = np.sqrt(
    mean_squared_error(test,forecast)
)

mae = mean_absolute_error(
    test,
    forecast
)

mape = mean_absolute_percentage_error(
    test,
    forecast
)*100

sarima_results = {

    "Model":"SARIMA",

    "Order":(0,1,0),

    "Seasonal Order":(1,1,0,12),

    "RMSE":rmse,

    "MAE":mae,

    "MAPE":mape

}

sarima_results

In [ ]:
future = fit.get_forecast(steps=48)

forecast = future.predicted_mean

confidence = future.conf_int(alpha=0.05)

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(
    series,
    label="Historical",
    lw=2
)

plt.plot(
    forecast.index,
    forecast,
    "r--",
    lw=2,
    label="Forecast"
)

plt.fill_between(
    forecast.index,
    confidence.iloc[:,0],
    confidence.iloc[:,1],
    alpha=0.25,
    color="red",
    label="95% Confidence Interval"
)

plt.title("GDP Forecast (48 Months)")
plt.grid(alpha=0.3)
plt.legend()

plt.show()

In [ ]:
forecast_df_GDP = forecast.copy()

confidence_df_GDP = confidence.copy()

confidence_df_GDP.columns = [
    "Lower GDP",
    "Upper GDP"
]

In [ ]:
print(forecast_df_GDP.head())
print(forecast_df_GDP.tail())

print(confidence_df_GDP.head())
print(confidence_df_GDP.tail())

# ||| **Repo Rate** |||

In [ ]:
series = pd.to_numeric(
    df["Repo_Rate"],
    errors="coerce"
).dropna()

series = series.asfreq("MS")

series.head()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(
    series,
    label="Repo Rate"
)

plt.plot(
    series.rolling(12).mean(),
    label="12M Rolling Mean"
)

plt.plot(
    series.rolling(12).std(),
    label="12M Rolling Std"
)

plt.title("Repo Rate")
plt.xlabel("Year")
plt.ylabel("Repo Rate (%)")
plt.grid(alpha=.3)
plt.legend()

plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(series)

print("ADF Statistic :",result[0])
print("P-value :",result[1])

print()

for k,v in result[4].items():
    print(k,":",v)

In [ ]:
series_diff = series.diff().dropna()

plt.figure(figsize=(15,4))

plt.plot(series_diff)

plt.axhline(0,color="red",ls="--")

plt.title("First Difference")

plt.grid(alpha=.3)

plt.show()

print(
    "ADF after First Difference:",
    adfuller(series_diff)[1]
)

In [ ]:
series_diff2 = series.diff().diff().dropna()

plt.figure(figsize=(15,4))

plt.plot(series_diff2)

plt.axhline(0,color="red",ls="--")

plt.title("Second Difference")

plt.grid(alpha=.3)

plt.show()

print(
    "ADF after Second Difference:",
    adfuller(series_diff2)[1]
)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf

if adfuller(series)[1] < 0.05:

    model_series = series

elif adfuller(series_diff)[1] < 0.05:

    model_series = series_diff

else:

    model_series = series_diff2

plot_acf(
    model_series,
    lags=30
)

plt.show()

plot_pacf(
    model_series,
    lags=30
)

plt.show()

In [ ]:
plot_acf(series_diff,lags = 30)

plt.show()

plot_pacf(series_diff,lags = 30)

plt.show()

In [ ]:
train = series.iloc[:-24]

test = series.iloc[-24:]

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

results = []

best_rmse = np.inf

best_model = None

best_order = None

d = 1       # change to 0 or 2 if required

for p in range(4):

    for q in range(4):

        try:

            model = ARIMA(
                train,
                order=(p,d,q)
            )

            fit = model.fit()

            forecast = fit.forecast(
                steps=len(test)
            )

            rmse = np.sqrt(
                mean_squared_error(
                    test,
                    forecast
                )
            )

            mae = mean_absolute_error(
                test,
                forecast
            )

            mape = mean_absolute_percentage_error(
                test,
                forecast
            )*100

            results.append({

                "Order":(p,d,q),

                "AIC":fit.aic,

                "BIC":fit.bic,

                "RMSE":rmse,

                "MAE":mae,

                "MAPE":mape

            })

            if rmse < best_rmse:

                best_rmse = rmse

                best_model = fit

                best_order = (p,d,q)

        except:

            pass

arima_table = pd.DataFrame(results)

arima_table.sort_values("RMSE")

In [ ]:
model = ARIMA(
    train,
    order=(1,1,2)
)

fit = model.fit()

print(fit.summary())

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(
    train,
    label="Train",
    lw=2
)

train_fit = fit.predict(
    start=d,
    end=len(train)-1,
    typ="levels"
)

plt.plot(
    train.index[d:],
    train_fit,
    label="Train Fitted",
    lw=2
)

plt.plot(
    test,
    label="Actual",
    lw=2
)

plt.plot(
    test.index,
    forecast,
    "r--",
    label="Forecast",
    lw=2
)

plt.title("ARIMA Forecast : Repo Rate")

plt.grid(alpha=.3)

plt.legend()

plt.show()

In [ ]:
residuals = fit.resid

plt.figure(figsize=(15,4))
plt.plot(residuals)
plt.axhline(
    0,
    color="red",
    ls="--"
)
plt.title("ARIMA Residuals : Repo Rate")
plt.grid(alpha=.3)
plt.show()

In [ ]:
plt.figure(figsize=(7,5))
plt.hist(
    residuals,
    bins=25,
    edgecolor="black"
)
plt.title("Residual Distribution : Repo Rate")
plt.grid(alpha=.3)
plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(
    residuals,
    lags=30
)
plt.title("ACF of Residuals : Repo Rate")
plt.show()

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

lb_test_results = acorr_ljungbox(
    residuals,
    lags=[10],
    return_df=True
)

print("Ljung-Box Test Results:")
print(lb_test_results)

In [ ]:
from statsmodels.stats.diagnostic import het_arch

arch_test_results = het_arch(residuals)

print("ARCH LM Test Results:")
print(f"LM Statistic : {arch_test_results[0]:.4f}")
print(f"P-value      : {arch_test_results[1]:.4f}")

In [ ]:
train_fit = fit.predict(
    start=1,
    end=len(train)-1,
    typ="levels"
)

In [ ]:
plt.plot(
    train.index[1:],
    train_fit,
    label="Train Fitted"
)

In [ ]:
from arch import arch_model

garch = arch_model(
    residuals,
    mean="Zero",
    vol="GARCH",
    p=1,
    q=1,
    dist="normal"
)

garch_fit = garch.fit(
    disp="off"
)

print(garch_fit.summary())

In [ ]:
volatility = garch_fit.conditional_volatility

plt.figure(figsize=(15,5))

plt.plot(
    volatility,
    lw=2
)

plt.title("Conditional Volatility")

plt.grid(alpha=.3)

plt.show()

In [ ]:
garch_forecast = garch_fit.forecast(
    horizon=48,
    reindex=False
)

future_sigma = np.sqrt(
    garch_forecast.variance.values[-1]
)

future_sigma

In [ ]:
future = fit.get_forecast(
    steps=48
)

forecast = future.predicted_mean

In [ ]:
lower = forecast - 1.96 * future_sigma

upper = forecast + 1.96 * future_sigma

In [ ]:
plt.figure(figsize=(15,6))

plt.plot(
    series,
    label="Historical",
    lw=2
)

plt.plot(
    forecast.index,
    forecast,
    "r--",
    lw=2,
    label="Forecast"
)

plt.fill_between(
    forecast.index,
    lower,
    upper,
    alpha=0.25,
    color="red",
    label="95% Uncertainty"
)

plt.title("ARIMA + GARCH Forecast : Repo Rate")

plt.grid(alpha=.3)

plt.legend()

plt.show()

In [ ]:
forecast_df_RepoRate = forecast.copy()

confidence_df_RepoRate = pd.DataFrame({

    "Lower Repo Rate": lower,

    "Upper Repo Rate": upper

})

forecast_df_RepoRate.head()

confidence_df_RepoRate.head()